# Introduction to Causal Inference: DoWhy and the Lalonde Dataset

Does a job training program actually cause participants to earn more, or do people who enroll in training simply differ from those who do not? This is the central challenge of **causal inference**: distinguishing genuine treatment effects from confounding differences between groups. A simple comparison of average earnings between participants and non-participants can be misleading if the two groups differ in age, education, or prior employment history.

**DoWhy** is a Python library that provides a principled, end-to-end framework for causal inference. It organizes the analysis into four explicit steps --- **Model, Identify, Estimate, Refute** --- each of which forces the analyst to state and test causal assumptions rather than hiding them inside a black-box estimator. In this tutorial, we apply DoWhy to the **Lalonde dataset**, a classic dataset from the National Supported Work (NSW) Demonstration program, to estimate how much the job training program increased participants' earnings in 1978.

**Learning objectives:**

- Understand DoWhy's four-step causal inference workflow (Model, Identify, Estimate, Refute)
- Define a causal graph that encodes domain knowledge about confounders
- Identify causal estimands from the graph using the backdoor criterion
- Estimate causal effects using multiple methods (regression adjustment, IPW, doubly robust, propensity score stratification, propensity score matching)
- Assess robustness of estimates using refutation tests

## Setup and imports

Install the required package if needed:

In [ ]:
!pip install dowhy -q

The following code imports all necessary libraries and sets configuration variables. We define the outcome, treatment, and covariate columns that will be used throughout the analysis.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression as SklearnLR
from dowhy import CausalModel
from dowhy.datasets import lalonde_dataset

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configuration
OUTCOME = "re78"
OUTCOME_LABEL = "Earnings in 1978 (USD)"
TREATMENT = "treat"
TREATMENT_LABEL = "Job Training (treat)"
COVARIATES = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

## Data loading: The Lalonde Dataset

The Lalonde dataset comes from the **National Supported Work (NSW) Demonstration**, a randomized employment program conducted in the 1970s in the United States. Eligible applicants --- mostly disadvantaged workers with limited employment histories --- were randomly assigned to receive job training (treatment) or not (control). The dataset records each participant's demographics, prior earnings, and post-program earnings in 1978. It has become a benchmark for testing causal inference methods because the random assignment provides a credible ground truth against which observational estimators can be compared.

DoWhy includes the Lalonde dataset directly, so we can load it with a single function call.

In [ ]:
df = lalonde_dataset()

# Convert boolean treatment to integer for DoWhy compatibility
df[TREATMENT] = df[TREATMENT].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment groups:")
print(df[TREATMENT].value_counts().sort_index().rename({0: "Control", 1: "Training"}))
print(f"\nOutcome ({OUTCOME}) summary:")
print(df[OUTCOME].describe().round(2))

The dataset contains 445 participants with 12 variables. The treatment is split into 185 individuals who received job training and 260 controls who did not. The outcome variable, real earnings in 1978 (`re78`), has a mean of \$5,301 but enormous variation (standard deviation of \$6,631), ranging from \$0 to \$60,308. The median (\$3,702) is well below the mean, indicating a right-skewed distribution --- many participants earned little or nothing while a few earned substantially more.

## Exploratory data analysis

### Outcome distribution by treatment group

Before any causal modeling, we compare the raw earnings distributions between training and control groups. If the training program had an effect, we expect to see higher average earnings in the training group --- but we cannot yet tell whether any difference is truly caused by the program or driven by pre-existing differences between the groups.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for group, label, color in [(0, "Control", STEEL_BLUE), (1, "Training", WARM_ORANGE)]:
    subset = df[df[TREATMENT] == group][OUTCOME]
    ax.hist(subset, bins=30, alpha=0.6, label=f"{label} (mean=${subset.mean():,.0f})",
            color=color, edgecolor="white")
ax.set_xlabel(OUTCOME_LABEL)
ax.set_ylabel("Count")
ax.set_title(f"Distribution of {OUTCOME_LABEL} by Treatment Group")
ax.legend()
plt.show()

Both distributions are heavily right-skewed, with a large spike near zero reflecting participants who had no earnings. The training group has a higher mean (\$6,349) compared to the control group (\$4,555), a raw difference of about \$1,794. However, both distributions overlap substantially, and the spike at zero is present in both groups, indicating that many participants struggled to find employment regardless of training.

### Covariate balance

In a randomized experiment, we expect the covariates to be balanced across treatment and control groups. Checking this balance is important: if the groups differ systematically in age, education, or prior earnings, any difference in the outcome could be driven by these confounders rather than the treatment itself.

In [ ]:
covariate_means = df.groupby(TREATMENT)[COVARIATES].mean()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(COVARIATES))
width = 0.35
ax.bar(x - width / 2, covariate_means.loc[0], width, label="Control",
       color=STEEL_BLUE, edgecolor="white")
ax.bar(x + width / 2, covariate_means.loc[1], width, label="Training",
       color=WARM_ORANGE, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(COVARIATES, rotation=45, ha="right")
ax.set_ylabel("Mean Value")
ax.set_title("Covariate Balance: Control vs Training Group")
ax.legend()
plt.show()

The demographic covariates (age, education, race, marital status) are relatively balanced between groups, which is expected from random assignment. However, the prior earnings variables (`re74` and `re75`) show noticeable differences: the control group has somewhat higher mean prior earnings. The sample is predominantly Black (83%), young (mean age 25.4), with low education (mean 10.2 years) and high rates of lacking a high school diploma (78%) --- reflecting the disadvantaged population targeted by the NSW program.

## The causal inference problem

### Why simple comparisons can mislead

A naive approach to estimating the treatment effect is to compute the difference in mean outcomes between the training and control groups. This gives us the **Average Treatment Effect (ATE)**:

$$\text{ATE}_{naive} = \bar{Y}_{treated} - \bar{Y}_{control}$$

While this is a natural starting point, it can be biased if the groups differ in ways that independently affect earnings. Even in a randomized experiment, finite-sample imbalances in covariates can introduce bias.

In [ ]:
mean_treated = df[df[TREATMENT] == 1][OUTCOME].mean()
mean_control = df[df[TREATMENT] == 0][OUTCOME].mean()
naive_ate = mean_treated - mean_control

print(f"Mean earnings (Training): ${mean_treated:,.2f}")
print(f"Mean earnings (Control):  ${mean_control:,.2f}")
print(f"Naive ATE (difference):   ${naive_ate:,.2f}")

The naive estimate suggests that training increases earnings by \$1,794 on average. But can we trust this number? Without adjusting for confounders, we cannot be sure how much of this difference is due to the training itself versus pre-existing differences between the groups. This is where DoWhy's structured framework helps --- it forces us to explicitly model our causal assumptions, identify the correct estimand, apply rigorous estimation methods, and test whether the results hold up under scrutiny.

## Step 1: Model --- Define the causal graph

The first step in DoWhy's framework is to encode our **domain knowledge** as a causal graph --- a Directed Acyclic Graph (DAG) that specifies which variables cause which. In our case, the covariates (age, education, race, prior earnings, etc.) are **common causes** of both treatment assignment and the outcome. Even in a randomized experiment, these confounders can affect outcomes and introduce finite-sample bias, so we include them in the model.

The causal structure we assume is:
- Each covariate (age, educ, black, hisp, married, nodegr, re74, re75) affects both treatment assignment and earnings
- Treatment (`treat`) affects the outcome (`re78`)
- No covariate is itself caused by the treatment (pre-treatment variables)

In [ ]:
# Visualize the assumed causal structure
fig, ax = plt.subplots(figsize=(10, 7))
confounders = COVARIATES
n_conf = len(confounders)

treatment_pos = (0.2, 0.5)
outcome_pos = (0.8, 0.5)
conf_positions = []
for i, c in enumerate(confounders):
    y = 0.9 - (i / (n_conf - 1)) * 0.8
    conf_positions.append((0.5, y))

for i, (cx, cy) in enumerate(conf_positions):
    ax.annotate("", xy=treatment_pos, xytext=(cx, cy),
                arrowprops=dict(arrowstyle="->", color="#cccccc", lw=1.0))
    ax.annotate("", xy=outcome_pos, xytext=(cx, cy),
                arrowprops=dict(arrowstyle="->", color="#cccccc", lw=1.0))

ax.annotate("", xy=outcome_pos, xytext=treatment_pos,
            arrowprops=dict(arrowstyle="->", color=WARM_ORANGE, lw=3.0))

for i, c in enumerate(confounders):
    cx, cy = conf_positions[i]
    ax.plot(cx, cy, "o", color=STEEL_BLUE, markersize=20, zorder=5)
    ax.text(cx + 0.06, cy, c, fontsize=9, va="center", ha="left", color=NEAR_BLACK)

ax.plot(*treatment_pos, "s", color=WARM_ORANGE, markersize=30, zorder=5)
ax.text(treatment_pos[0], treatment_pos[1] - 0.07, "treat", fontsize=11,
        ha="center", fontweight="bold", color=NEAR_BLACK)
ax.plot(*outcome_pos, "s", color=TEAL, markersize=30, zorder=5)
ax.text(outcome_pos[0], outcome_pos[1] - 0.07, "re78", fontsize=11,
        ha="center", fontweight="bold", color=NEAR_BLACK)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Causal Graph: NSW Job Training Program", fontsize=14)
ax.axis("off")
plt.show()

The DAG makes our assumptions explicit: the eight covariates (blue circles) are confounders that affect both treatment assignment and earnings. The orange arrow from `treat` to `re78` is the causal effect we want to estimate. By stating these assumptions as a graph, DoWhy can automatically determine which variables need to be adjusted for and which estimation strategies are valid.

Now we create the `CausalModel` in DoWhy, specifying the treatment, outcome, and common causes:

In [ ]:
model = CausalModel(
    data=df,
    treatment=TREATMENT,
    outcome=OUTCOME,
    common_causes=COVARIATES,
)
print("CausalModel created successfully.")

The `CausalModel` object stores the data, the causal graph, and metadata about treatment, outcome, and confounders. DoWhy will use this graph in the next step to determine the correct adjustment strategy.

## Step 2: Identify --- Find the causal estimand

With the causal graph defined, DoWhy uses graph theory to **identify** the causal estimand --- the mathematical expression that, if computed correctly, equals the true causal effect. This step determines *whether* the effect is identifiable from the data given our assumptions, and *what* variables we need to condition on.

The key concept here is the **backdoor criterion**: if we can block all "backdoor paths" from treatment to outcome (paths that go through confounders), then we can identify the causal effect by conditioning on those confounders.

In [ ]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

DoWhy identifies the **backdoor estimand** as the primary identification strategy, expressing the causal effect as the derivative of the conditional expectation of earnings with respect to treatment, conditioning on all eight covariates. The critical assumption is **unconfoundedness** --- there are no unmeasured confounders beyond the ones we specified. DoWhy also checks for instrumental variable and front-door estimands but finds none applicable, which is expected given our graph structure.

## Step 3: Estimate --- Compute the causal effect

With the estimand identified, we now apply statistical methods to compute the actual causal effect estimate. DoWhy supports multiple estimation methods, each with different assumptions and properties. We compare five approaches to see how robust the estimate is across methods.

### Method 1: Regression Adjustment

The simplest approach --- also called **regression adjustment** --- adjusts for confounders by including them as covariates in a linear regression. The treatment effect is the coefficient on treatment after controlling for covariates. This assumes a linear relationship between covariates and the outcome, which may be restrictive but provides a transparent baseline.

In [ ]:
estimate_ra = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
)
print(f"Estimated ATE (Regression Adjustment): ${estimate_ra.value:,.2f}")

The regression adjustment estimate is \$1,676, slightly lower than the naive difference of \$1,794. The reduction from \$1,794 to \$1,676 reflects the covariate adjustment --- part of the raw earnings gap was attributable to differences in observable characteristics between groups. After accounting for age, education, race, and prior earnings, the estimated treatment effect shrinks by about \$118.

### Method 2: Inverse Probability Weighting (IPW)

IPW takes a fundamentally different approach from regression adjustment. Instead of modeling the outcome, it models the **treatment assignment mechanism**. Each observation is weighted by the inverse of its probability of receiving the treatment it actually received (the **propensity score**). This reweighting creates a "pseudo-population" in which treatment assignment is independent of the observed confounders.

The IPW estimator is:

$$\hat{\tau}_{IPW} = \frac{1}{n} \sum_{i=1}^{n} \left[ \frac{T_i Y_i}{\hat{e}(X_i)} - \frac{(1 - T_i) Y_i}{1 - \hat{e}(X_i)} \right]$$

where $\hat{e}(X_i)$ is the estimated propensity score for individual $i$.

In [ ]:
estimate_ipw = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    method_params={"weighting_scheme": "ips_weight"},
)
print(f"Estimated ATE (IPW): ${estimate_ipw.value:,.2f}")

The IPW estimate of \$1,559 is the lowest among all methods. IPW is sensitive to extreme propensity scores --- when some individuals have very high or very low probabilities of treatment, their weights become large and can dominate the estimate. The difference from the regression adjustment (\$1,676 vs \$1,559) reflects the fact that IPW makes no assumptions about the outcome model, relying entirely on correct specification of the propensity score model.

### Method 3: Doubly Robust (AIPW)

The **doubly robust** estimator --- also called **Augmented Inverse Probability Weighting (AIPW)** --- combines both regression adjustment and IPW into a single estimator. The key advantage is that the estimate is consistent if *either* the outcome model *or* the propensity score model is correctly specified (hence "doubly robust").

The AIPW estimator is:

$$\hat{\tau}_{DR} = \frac{1}{n} \sum_{i=1}^{n} \left[ \hat{\mu}_1(X_i) - \hat{\mu}_0(X_i) + \frac{T_i (Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1 - T_i)(Y_i - \hat{\mu}_0(X_i))}{1 - \hat{e}(X_i)} \right]$$

where $\hat{\mu}_1(X_i)$ and $\hat{\mu}_0(X_i)$ are the predicted outcomes under treatment and control, and $\hat{e}(X_i)$ is the propensity score.

We implement the AIPW estimator manually rather than using DoWhy's built-in `backdoor.doubly_robust` method, which has a known compatibility issue with recent scikit-learn versions. The manual implementation also makes the estimator's two-component structure --- propensity score model plus outcome model --- fully transparent.

In [ ]:
# Doubly Robust (AIPW) — manual implementation
ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(df[COVARIATES], df[TREATMENT])
ps = ps_model.predict_proba(df[COVARIATES])[:, 1]

outcome_model_1 = SklearnLR().fit(df[df[TREATMENT] == 1][COVARIATES], df[df[TREATMENT] == 1][OUTCOME])
outcome_model_0 = SklearnLR().fit(df[df[TREATMENT] == 0][COVARIATES], df[df[TREATMENT] == 0][OUTCOME])

mu1 = outcome_model_1.predict(df[COVARIATES])
mu0 = outcome_model_0.predict(df[COVARIATES])
T = df[TREATMENT].values
Y = df[OUTCOME].values

dr_ate = np.mean(
    (mu1 - mu0)
    + T * (Y - mu1) / ps
    - (1 - T) * (Y - mu0) / (1 - ps)
)
print(f"Estimated ATE (Doubly Robust): ${dr_ate:,.2f}")

The doubly robust estimate of \$1,620 falls between the regression adjustment (\$1,676) and IPW (\$1,559) estimates. This makes intuitive sense: the AIPW estimator "averages" information from both the outcome model and the propensity score model. In practice, the doubly robust estimator is often the preferred choice because it provides insurance against misspecification of either component model.

### Method 4: Propensity Score Stratification

Propensity score stratification works by first estimating each individual's probability of receiving treatment given their covariates (the **propensity score**), then dividing the sample into strata with similar propensity scores. Within each stratum, treated and control individuals are more comparable, so the within-stratum treatment effect is less biased. The overall ATE is a weighted average of the stratum-specific effects.

In [ ]:
estimate_ps_strat = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_stratification",
    method_params={"num_strata": 5, "clipping_threshold": 5},
)
print(f"Estimated ATE (PS Stratification): ${estimate_ps_strat.value:,.2f}")

Propensity score stratification with 5 strata estimates the ATE at \$1,617, very close to the doubly robust estimate (\$1,620). The stratification approach is more flexible than linear regression because it does not impose a functional form on the outcome-covariate relationship. The estimate is in the same ballpark as the regression result, which is reassuring --- multiple methods agree that the training effect is in the \$1,550--\$1,700 range.

### Method 5: Propensity Score Matching

Propensity score matching pairs each treated individual with one or more control individuals who have similar propensity scores. The treatment effect is estimated by comparing outcomes within matched pairs. This is conceptually the most intuitive approach --- it mimics what we would see if we could directly compare individuals who are identical except for their treatment status.

In [ ]:
estimate_ps_match = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
)
print(f"Estimated ATE (PS Matching): ${estimate_ps_match.value:,.2f}")

Propensity score matching estimates the ATE at \$1,736, the highest of the five adjusted estimates and closest to the naive difference. Matching tends to give slightly different results because it uses only the closest comparisons rather than the full sample. The fact that all five methods produce estimates between \$1,559 and \$1,736 provides strong evidence that the treatment effect is real and robust to the choice of estimation method.

## Step 4: Refute --- Test robustness

The final and perhaps most valuable step in DoWhy's framework is **refutation** --- systematically testing whether the estimated causal effect is robust to violations of our assumptions. DoWhy provides several built-in refutation tests, each probing a different potential weakness.

### Placebo Treatment Test

The placebo test replaces the actual treatment with a randomly permuted version. If our estimate is truly capturing a causal effect, this fake treatment should produce an effect near zero.

In [ ]:
refute_placebo = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=100,
)
print(refute_placebo)

The placebo treatment test produces a new effect close to zero, dramatically smaller than the original estimate of \$1,676. The high p-value indicates that the original estimate is well above what we would expect from a random treatment assignment. This is strong evidence that the estimated effect is not an artifact of the model or data structure.

### Random Common Cause Test

This test adds a randomly generated confounder to the model and checks whether the estimate changes. If our model is correctly specified, adding a random variable should not significantly alter the result.

In [ ]:
refute_random = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="random_common_cause",
    num_simulations=100,
)
print(refute_random)

Adding a random common cause barely changes the estimate --- the new effect is nearly identical to the original \$1,676. This confirms that the model is not overly sensitive to the specific set of confounders included.

### Data Subset Test

The data subset test re-estimates the effect on random 80% subsamples of the data. If the estimate is robust, it should remain similar across different subsets.

In [ ]:
refute_subset = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=100,
)
print(refute_subset)

The data subset refuter produces a mean effect close to the full-sample estimate of \$1,676, indicating that the estimate is stable across subsets and does not depend on a handful of outlier observations.

## Comparing all estimates

To visualize how all estimation approaches compare, we plot the ATE estimates side by side. Consistent estimates across different methods strengthen confidence in the causal conclusion.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
methods = ["Naive\n(Diff. in Means)", "Regression\nAdjustment", "IPW",
           "Doubly Robust\n(AIPW)", "PS\nStratification", "PS\nMatching"]
estimates = [naive_ate, estimate_ra.value, estimate_ipw.value,
             dr_ate, estimate_ps_strat.value, estimate_ps_match.value]
colors = ["#999999", STEEL_BLUE, WARM_ORANGE, TEAL, "#8b5cf6", "#f59e0b"]

bars = ax.barh(methods, estimates, color=colors, edgecolor="white", height=0.6)
for bar, val in zip(bars, estimates):
    offset = 50 if val >= 0 else -50
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"${val:,.0f}", va="center", ha=ha, fontsize=10, color=NEAR_BLACK)

ax.axvline(0, color="black", linewidth=0.5, linestyle="--")ax.set_xlabel("Estimated Average Treatment Effect (USD)")ax.set_title("Causal Effect Estimates: NSW Job Training on 1978 Earnings")plt.show()

All six methods produce positive estimates between \$1,559 and \$1,794, indicating that the NSW job training program increased participants' 1978 earnings by roughly \$1,550--\$1,800. The adjusted methods cluster between \$1,559 and \$1,736, suggesting that about \$40--\$235 of the naive estimate was due to confounding rather than the treatment. Notably, the five adjusted estimators represent three distinct paradigms --- outcome modeling (regression adjustment), treatment modeling (IPW), and a hybrid approach (doubly robust) --- yet all converge on a similar range, reinforcing confidence in the causal conclusion.

## Summary table

In [ ]:
results_df = pd.DataFrame({
    "Method": ["Naive (Diff. in Means)", "Regression Adjustment", "IPW",
               "Doubly Robust (AIPW)", "PS Stratification", "PS Matching"],
    "Estimated ATE (USD)": [f"${naive_ate:,.0f}", f"${estimate_ra.value:,.0f}",
                            f"${estimate_ipw.value:,.0f}", f"${dr_ate:,.0f}",
                            f"${estimate_ps_strat.value:,.0f}", f"${estimate_ps_match.value:,.0f}"],
    "Notes": ["No covariate adjustment", "Models outcome, assumes linearity",
              "Models treatment assignment", "Models both outcome and treatment",
              "5 strata, flexible", "Nearest-neighbor matching"],
})print(results_df.to_string(index=False))

## Discussion

The Lalonde dataset provides a compelling case study for DoWhy's four-step framework. Each step serves a distinct purpose: the **Model** step forces us to articulate our causal assumptions as a graph, the **Identify** step uses graph theory to determine the correct adjustment formula, the **Estimate** step applies statistical methods to compute the effect, and the **Refute** step probes whether the result withstands scrutiny.

The estimated ATE ranges from \$1,559 (IPW) to \$1,736 (PS matching), with the doubly robust estimate at \$1,620. On average, participants who received job training earned roughly \$1,600 more in 1978 than they would have without training. On a base of \$4,555 for the control group, this represents roughly a 34--38% increase in earnings --- a substantial effect for a disadvantaged population with very low baseline earnings.

The five estimators span three estimation paradigms: outcome modeling (regression adjustment), treatment modeling (IPW), and a hybrid approach (doubly robust/AIPW). Their convergence on the \$1,550--\$1,800 range is particularly compelling because each paradigm relies on different modeling assumptions.

The key strength of DoWhy over ad-hoc statistical approaches is transparency. The causal graph makes assumptions visible and debatable. The identification step formally checks whether the effect is estimable. Multiple estimation methods let us assess robustness. And refutation tests provide automated sanity checks that would otherwise require expert judgment.

## Takeaways

- **DoWhy's four-step workflow** (Model, Identify, Estimate, Refute) makes causal assumptions explicit and testable, rather than hiding them inside a black-box estimator.
- **The NSW job training program increased 1978 earnings by approximately \$1,550--\$1,800**, a 34--38% gain over the control group mean of \$4,555.
- **Five estimation methods** --- regression adjustment, IPW, doubly robust, PS stratification, and PS matching --- all produce positive, consistent estimates, strengthening confidence in the causal conclusion.
- **The doubly robust (AIPW) estimator** (\$1,620) is the most credible single estimate because it remains consistent if either the outcome model or the propensity score model is misspecified.
- **IPW and regression adjustment represent two complementary paradigms**: modeling treatment assignment (\$1,559) vs. modeling the outcome (\$1,676). Their divergence quantifies sensitivity to modeling choices.
- **Refutation tests confirm robustness** --- the placebo test reduced the effect from \$1,676 to just \$62, ruling out statistical artifacts.
- **Causal graphs encode domain knowledge as testable assumptions**; the backdoor criterion then determines which variables must be conditioned on for valid causal estimation.
- **Next step**: apply DoWhy to an observational comparison group (e.g., PSID or CPS) where confounding is stronger and the choice of estimator matters more.

## Exercises

1. **Change the number of strata.** Re-run the propensity score stratification with `num_strata=10` and `num_strata=20`. How does the ATE estimate change? What are the tradeoffs of using more vs. fewer strata with a sample of only 445 observations?

2. **Add an additional refutation test.** DoWhy supports a `bootstrap_refuter` that re-estimates the effect on bootstrap samples. Implement this refuter and compare its results to the data subset refuter. Are the conclusions similar?

3. **Estimate effects for subgroups.** Split the dataset by `black` (race indicator) and estimate the ATE separately for each subgroup using DoWhy. Does the job training program have a different effect for Black vs. non-Black participants? What might explain any differences you observe?

## References

1. [DoWhy --- Python Library for Causal Inference (PyWhy)](https://www.pywhy.org/dowhy/)
2. [LaLonde, R. (1986). Evaluating the Econometric Evaluations of Training Programs. American Economic Review, 76(4), 604--620.](https://www.jstor.org/stable/1806062)
3. [Dehejia, R. & Wahba, S. (1999). Causal Effects in Nonexperimental Studies: Reevaluating the Evaluation of Training Programs. JASA, 94(448), 1053--1062.](https://doi.org/10.1080/01621459.1999.10473858)
4. [Sharma, A. & Kiciman, E. (2020). DoWhy: An End-to-End Library for Causal Inference. arXiv:2011.04216.](https://arxiv.org/abs/2011.04216)
5. [Horvitz, D. G. & Thompson, D. J. (1952). A Generalization of Sampling Without Replacement from a Finite Universe. JASA, 47(260), 663--685.](https://doi.org/10.1080/01621459.1952.10483446)
6. [Robins, J. M., Rotnitzky, A. & Zhao, L. P. (1994). Estimation of Regression Coefficients When Some Regressors Are Not Always Observed. JASA, 89(427), 846--866.](https://doi.org/10.1080/01621459.1994.10476818)
7. [Nita, C. J. Causal Inference with Python --- Introduction to DoWhy. Medium.](https://medium.com/@chrisjames.nita/causal-inference-with-python-introduction-to-dowhy-ff5799e48985)